In [1]:
from openai import OpenAI

In [2]:
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

In [4]:
client = OpenAI()

In [5]:
prompt_template = """
For the query below create a response that has JSON format with the pair
"response": ... using the response to the user query as the value

{user_query}
"""

In [6]:
query_dict = {"user_query": "what is the capital of France?"}

In [7]:
from graphrunner.llmclasses import LLMCall, OpenAI_response_fn

ModuleNotFoundError: No module named 'graphrunner'

In [8]:
response_fn = OpenAI_response_fn(client)

In [9]:
llm = LLMCall(
    prompt_template=prompt_template,
    model="gpt-5.2",
    response_fn=response_fn
)

In [10]:
response = llm(query_dict)

In [11]:
response

{'raw_output': '{"response":"Paris."}',
 'usage': {'input_tokens': 44,
  'input_tokens_details': {'cached_tokens': 0},
  'output_tokens': 10,
  'output_tokens_details': {'reasoning_tokens': 0},
  'total_tokens': 54},
 'response': 'Paris.',
 'message_history': [{'role': 'user',
   'content': 'what is the capital of France?'},
  {'role': 'assistant', 'content': '{"response":"Paris."}'}]}

In [13]:
callable(client.responses.create)

True

In [14]:
message_pairs = [("What is the capital of France?", "Paris"), ("What is a notable landmark there?", "The Eiffel Tower")]

In [18]:
message_history = [
    {"role" : "user", "content": "What is the capital of France"},
    {"role" : "assistant", "content": "Paris"},
    {"role": "user", "content": "What is a notable landmark there?"},
    {"role": "assistant", "content" : "The Eiffel Tower"}
]

In [16]:
def dict_to_str(hist_dict):
    return f"{hist_dict['role']} : {hist_dict['content']}"

In [17]:
def message_hist_to_str(history):
    return "\n".join([dict_to_str(hist_dict) for hist_dict in history])

In [20]:
print(message_hist_to_str(message_history))

user : What is the capital of France
assistant : Paris
user : What is a notable landmark there?
assistant : The Eiffel Tower


In [21]:
print(message_hist_to_str([]))

In [22]:
message_history[-20:]

[{'role': 'user', 'content': 'What is the capital of France'},
 {'role': 'assistant', 'content': 'Paris'},
 {'role': 'user', 'content': 'What is a notable landmark there?'},
 {'role': 'assistant', 'content': 'The Eiffel Tower'}]

In [27]:
message_hist_to_str([]) == ''

True

In [28]:
if message_hist_to_str([]):
    print("true")
else:
    print("false")

false


In [15]:
query_dict_2 = {"user_query" : "What is a notable landmark there?", "message_history": response['message_history']}

In [16]:
llm(query_dict_2)

{'raw_output': '{"response":"A notable landmark in Paris is the Eiffel Tower."}',
 'usage': {'input_tokens': 60,
  'input_tokens_details': {'cached_tokens': 0},
  'output_tokens': 18,
  'output_tokens_details': {'reasoning_tokens': 0},
  'total_tokens': 78},
 'response': 'A notable landmark in Paris is the Eiffel Tower.',
 'message_history': [{'role': 'user',
   'content': 'what is the capital of France?'},
  {'role': 'assistant', 'content': '{"response":"Paris."}'},
  {'role': 'user', 'content': 'What is a notable landmark there?'},
  {'role': 'assistant',
   'content': '{"response":"A notable landmark in Paris is the Eiffel Tower."}'}]}

In [17]:
from graphrunner.nodes import GraphNode

In [18]:
node = GraphNode(
    func = llm,
    name = "llmnode"
)

In [19]:
node.execute(query_dict)

{'raw_output': '{\n  "response": "Paris."\n}',
 'usage': {'input_tokens': 44,
  'input_tokens_details': {'cached_tokens': 0},
  'output_tokens': 13,
  'output_tokens_details': {'reasoning_tokens': 0},
  'total_tokens': 57},
 'response': 'Paris.',
 'message_history': [{'role': 'user',
   'content': 'what is the capital of France?'},
  {'role': 'assistant', 'content': '{\n  "response": "Paris."\n}'}]}

In [21]:
from graphrunner.runner import GraphRunner

In [24]:
runner = GraphRunner(nodes=[node], start_node="llmnode")

In [25]:
runner.execute(query_dict)

{'raw_output': '{"response":"Paris."}',
 'usage': {'input_tokens': 44,
  'input_tokens_details': {'cached_tokens': 0},
  'output_tokens': 10,
  'output_tokens_details': {'reasoning_tokens': 0},
  'total_tokens': 54},
 'response': 'Paris.',
 'message_history': [{'role': 'user',
   'content': 'what is the capital of France?'},
  {'role': 'assistant', 'content': '{"response":"Paris."}'}]}

In [26]:
runner.execute({"user_query": "What is a landmark there?"})

{'raw_output': '{"response":"A famous landmark there is the Eiffel Tower."}',
 'usage': {'input_tokens': 59,
  'input_tokens_details': {'cached_tokens': 0},
  'output_tokens': 17,
  'output_tokens_details': {'reasoning_tokens': 0},
  'total_tokens': 76},
 'response': 'A famous landmark there is the Eiffel Tower.',
 'message_history': [{'role': 'user',
   'content': 'what is the capital of France?'},
  {'role': 'assistant', 'content': '{"response":"Paris."}'},
  {'role': 'user', 'content': 'What is a landmark there?'},
  {'role': 'assistant',
   'content': '{"response":"A famous landmark there is the Eiffel Tower."}'}]}